In [3]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

sns.set_style("whitegrid")


In [4]:
customers = pd.read_csv("data/customers.csv")
orders = pd.read_csv("data/orders.csv")
tickets = pd.read_csv("data/support_tickets.csv")
web = pd.read_csv("data/web_events_snapshot.csv")
churn = pd.read_csv("data/churn_labels.csv")
campaign = pd.read_csv("data/intervention_history.csv")

EmptyDataError: No columns to parse from file

In [ ]:
%pip install pandas numpy matplotlib seaborn

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 10.8 MB 3.4 MB/s eta 0:00:01
     |████████████████████████████████| 5.3 MB 3.8 MB/s eta 0:00:01
     |████████████████████████████████| 7.8 MB 3.4 MB/s eta 0:00:01
     |████████████████████████████████| 294 kB 5.3 MB/s eta 0:00:01
     |████████████████████████████████| 349 kB 32.9 MB/s eta 0:00:01
     |████████████████████████████████| 510 kB 27.5 MB/s eta 0:00:01
     |████████████████████████████████| 2.9 MB 19.8 MB/s eta 0:00:01     |██████████████████████████████▏ | 2.7 MB 19.8 MB/s eta 0:00:01
     |████████████████████████████████| 122 kB 5.2 MB/s eta 0:00:01
     |████████████████████████████████| 4.7 MB 3.5 MB/s eta 0:00:011     |███████████▎                    | 1.7 MB 19.0 MB/s eta 0:00:01
     |████████████████████████████████| 249 kB 5.9 MB/s eta 0:00:01
     |████████████████████████████████| 64 kB 9.6 MB/s  eta 0:00:01
You should consider upgrading via

In [ ]:
datasets = {
    "customers": customers,
    "orders": orders,
    "tickets": tickets,
    "web": web,
    "churn": churn,
    "campaign": campaign
}

for name, df in datasets.items():
    print("="*60)
    print(name)
    print(df.shape)
    print(df.info())

In [ ]:
for name, df in datasets.items():

    missing = df.isnull().sum()

    missing = missing[missing > 0]

    if len(missing) > 0:

        plt.figure(figsize=(8,4))

        missing.sort_values().plot(kind="barh")

        plt.title(f"Missing Values - {name}")

        plt.show()

In [ ]:
dup_orders = orders[
    orders["order_id"].str.contains("_DUP", na=False)
]

print("Duplicate-like Orders")
print(dup_orders.shape)

dup_orders.head()

In [ ]:
plt.figure(figsize=(10,5))

sns.boxplot(
    x=orders["gross_amount"]
)

plt.title("Gross Amount Outliers")

plt.show()

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(
    x="churn_next_60d",
    data=churn
)

plt.title("Churn Distribution")

plt.show()

print(
    churn["churn_next_60d"]
    .value_counts(normalize=True)
)

In [ ]:
orders_per_customer = (
    orders.groupby("customer_id")
    .size()
)

plt.figure(figsize=(10,5))

sns.histplot(
    orders_per_customer,
    bins=30
)

plt.title("Orders Per Customer")

plt.show()

In [ ]:
df = customers.merge(
    churn,
    on="customer_id"
)

loyalty_churn = pd.crosstab(
    df["loyalty_tier"],
    df["churn_next_60d"],
    normalize="index"
)

loyalty_churn.plot(
    kind="bar",
    stacked=True,
    figsize=(8,5)
)

plt.title("Loyalty Tier vs Churn")

plt.show()

In [ ]:
ticket_count = (
    tickets.groupby("customer_id")
    .size()
    .reset_index(name="ticket_count")
)

temp = churn.merge(
    ticket_count,
    on="customer_id",
    how="left"
)

temp["ticket_count"] = (
    temp["ticket_count"]
    .fillna(0)
)

sns.boxplot(
    x="churn_next_60d",
    y="ticket_count",
    data=temp
)

plt.title(
    "Support Tickets vs Churn"
)

plt.show()

In [ ]:
ratings = (
    orders.groupby("customer_id")["rating"]
    .mean()
    .reset_index()
)

temp = churn.merge(
    ratings,
    on="customer_id"
)

sns.boxplot(
    x="churn_next_60d",
    y="rating",
    data=temp
)

plt.title(
    "Average Rating vs Churn"
)

plt.show()

In [ ]:
temp = web.merge(
    churn,
    on="customer_id"
)

sns.boxplot(
    x="churn_next_60d",
    y="sessions_30d",
    data=temp
)

plt.title(
    "Sessions vs Churn"
)

plt.show()

In [ ]:
temp = campaign.merge(
    churn,
    on="customer_id"
)

campaign_table = pd.crosstab(
    temp["last_campaign_received"],
    temp["churn_next_60d"],
    normalize="index"
)

campaign_table.plot(
    kind="bar",
    stacked=True,
    figsize=(10,5)
)

plt.title(
    "Campaign vs Churn"
)

plt.show()